# NOR Analysis Pipeline

This notebook provides a modular, accessible workflow for analyzing the data.

In [ ]:
import sys
import os
# Add the parent directory to sys.path so we can import the backend files
sys.path.append(os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Import the core data extractor
from data_extractor import extract_all_data

# Import existing plotting functions from backend
from plotting import (
    event_rate,
    plot_dual_trace, plot_combines, plot_combined_auc, plot_events_per_second, 
    plot_total_events, plot_auc_bar, plot_combined_traces, plot_combined_rate, 
    plot_combined_events, plot_combined, histo, auc_rate_scatter, stationary_locomotion, 
    save_traces_per_mouse, get_random_non_interaction_trace, plot_trace_comparison, plot_interval_auc
)

## Data Extraction
> **Note:** This is the only cell that takes significant time to run, as it processes and loads all the data into memory. Once completed, all subsequent visualization cells can be run instantly.

In [ ]:
# Parameters Setup
action_input = "entrance" # Entrance or Exit?
zscore_input = "y"        # Z_score? Y or No
save_file_input = "y"     # Save? Y or No

action = action_input.strip().lower() == "entrance"
zscore = zscore_input.strip().lower() == "y"
save_file = save_file_input.strip().lower() == "y"

# How many total seconds from 10 seconds before interaction do we want
trunc_time = 20 

# Extract data
extracted_data = extract_all_data(action=action, zscore=zscore, save_file=save_file, trunc_time=trunc_time)

# Unpack data structures
all_traces = extracted_data['all_traces']
all_amplitudes = extracted_data['all_amplitudes']
all_durations = extracted_data['all_durations']
all_widths = extracted_data['all_widths']
all_aucs = extracted_data['all_aucs']
all_events = extracted_data['all_events']
all_rates = extracted_data['all_rates']
all_aucs_rate = extracted_data['all_aucs_rate']
all_aucs_pre_event = extracted_data['all_aucs_pre_event']
all_aucs_post_event = extracted_data['all_aucs_post_event']
all_aucrate_pre_event = extracted_data['all_aucrate_pre_event']
all_aucrate_post_event = extracted_data['all_aucrate_post_event']
all_width_pre_event = extracted_data['all_width_pre_event']
all_width_post_event = extracted_data['all_width_post_event']
all_amplitude_pre_event = extracted_data['all_amplitude_pre_event']
all_amplitude_post_event = extracted_data['all_amplitude_post_event']
all_duration_pre_event = extracted_data['all_duration_pre_event']
all_duration_post_event = extracted_data['all_duration_post_event']
all_mouse_ids = extracted_data['all_mouse_ids']
all_non_interaction_means = extracted_data['all_non_interaction_means']
time_axis = extracted_data['time_axis']

print("Data extraction complete! You can now run the visualization cells.")

## Visualizations
Run the individual cells below to generate specific plots or export data.

### Event-related Summaries (Pre/Post)

In [ ]:
group = 'NOR'
condition = 'NOV' # or 'FAM'

for event in ["aucs", "aucrate", "width", "amplitude", "duration"]:
    # Plot Pre-Event
    plot_combines(
        eval(f"all_{event}_pre_event"),
        f"{event}(Pre-Event)",
        max_value=None,
        group=group,
        condition=condition,
        mouse_ids=all_mouse_ids[group],
        title=f"{group} Pre-event {event}",
        save_path=False,
        action=action,
        combined=True
    )

    # Plot Post-Event
    plot_combines(
        eval(f"all_{event}_post_event"),
        f"{event}(Post-Event)",
        max_value=None,
        group=group,
        condition=condition,
        mouse_ids=all_mouse_ids[group],
        title=f"{group} Post-event {event}",
        save_path=False,
        action=action,
        combined=True
    )
plt.show()

### AUC Interval Plot

In [ ]:
plot_interval_auc(
    all_traces,
    group,
    time_axis,
    zscore,
    action,
    save_path=False
)
plt.show()

### AUC Histogram

In [ ]:
histo(
    all_aucs, 
    group, 
    condition, 
    zscore, 
    trunc_time, 
    title=f"AUC Histogram {group} {condition}", 
    save_path=False, 
    action=action
)
plt.show()

### AUC vs Rate Scatter Plot

In [ ]:
auc_rate_scatter(
    all_aucs_rate, 
    group, 
    epoch="peri-event", 
    unit="AUC/MIN", 
    save_path=False, 
    action=action
)
plt.show()

### Combined Traces Plot

In [ ]:
# Example: Plot combined traces
# Ensure you define size if needed, e.g., size=(6,4)
plot_combined_traces(
    all_traces,
    group,
    time_axis,
    zscore,
    action,
    size=(6,4),
    save_path=False
)
plt.show()

### Simple Event Rate Bar Chart

In [ ]:
# Simple bar chart of event rates for a single condition (e.g., NOV)
event_rate(
    all_rates, 
    group, 
    condition, 
    action=action, 
    save_path=False
)
plt.show()

### Save Traces to CSV

In [ ]:
# Ensure you set save_path=True if you want to write the file
save_traces_per_mouse(
    all_traces, 
    group, 
    time_axis, 
    all_mouse_ids[group], 
    zscore, 
    action, 
    save_path=save_file
)
print("Traces saved if save_file was True.")